In [5]:
from abc import ABC,abstractmethod # type: ignore
from typing import Any



class BaseTool(ABC):

    @property
    @abstractmethod
    def name(self)->str:  # type: ignore
        pass

    @property
    @abstractmethod
    def description(self)->str:  # type: ignore
        pass


    @abstractmethod
    def execute(self,action:str,**kwargs)->Any:
        pass



In [6]:
# * File Manager Tool for Agent

import os
import shutil
from pathlib import Path
from typing import Any, Dict, List
# from notebooks.base_tool import BaseTool


class FileManager(BaseTool):
    """Tools for managing local files and directories.

    All paths may be absolute or relative to the current working directory.
    """

    def __init__(self) -> None:
        """Initialize the file manager."""
        pass

    
    @property
    def name(self):
        return "file"
    
    @property
    def description(self):
        return "Manage files and directories"
        
    def execute(self, action: str, **kwargs):

        actions = {

            "create_file": self.create_file,

            "read_file": self.read_file,

            "write_file": self.write_file,

            "append_file": self.append_file,

            "delete_file": self.delete_file,

            "create_directory": self.create_directory,

            "delete_directory": self.delete_directory,

            "move": self.move,

            "copy": self.copy,

            "rename": self.rename,

            "exists": self.exists,

            "list_directories": self.list_directories,

            "search": self.search,

            "metadata": self.get_metadata,

            "open_file": self.open_file,

            "open_directory": self.open_directory,
        }

        if action not in actions:
            raise ValueError(f"Unknown action: {action}")

        return actions[action](**kwargs)

    #^ ==========================================================
    #^ 1) File Related Tools
    #^ ==========================================================

    def create_file(self, file_name: str) -> str:
        """Create an empty file.

        Parent directories are created automatically.

        Args:
            file_name: File path to create.

        Returns:
            Status message.
        """
        path = Path(file_name)

        path.parent.mkdir(parents=True, exist_ok=True)

        if path.exists():
            return f"File already exists: {path}"

        path.touch()

        return f"File created successfully: {path}"

    def read_file(self, file_name: str):
        """Read a file.

        Returns text for UTF-8 files, otherwise returns raw bytes.

        Args:
            file_name: File path.

        Returns:
            str | bytes: File contents.
        """
        path = Path(file_name)

        if not path.exists():
            raise FileNotFoundError(path)

        try:
            return path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            return path.read_bytes()

    def write_file(self, file_name: str, content) -> str:
        """Overwrite a file.

        Creates the file if it does not exist.

        Args:
            file_name: Destination file path.
            content: Text (str) or binary (bytes) data.

        Returns:
            Status message.
        """
        path = Path(file_name)

        path.parent.mkdir(parents=True, exist_ok=True)

        if isinstance(content, bytes):
            path.write_bytes(content)
        else:
            path.write_text(str(content), encoding="utf-8")

        return f"Written successfully: {path}"

    def append_file(self, file_name: str, content) -> str:
        """Append data to a file.

        Creates the file if it does not exist.

        Args:
            file_name: File path.
            content: Text (str) or binary (bytes) data.

        Returns:
            Status message.
        """
        path = Path(file_name)

        path.parent.mkdir(parents=True, exist_ok=True)

        if isinstance(content, bytes):
            with open(path, "ab") as f:
                f.write(content)
        else:
            with open(path, "a", encoding="utf-8") as f:
                f.write(str(content))

        return f"Appended successfully: {path}"

    def delete_file(self, file_name: str) -> str:
        """Delete a file.

        Args:
            file_name: File path.

        Returns:
            Status message.
        """
        path = Path(file_name)

        if not path.exists():
            raise FileNotFoundError(path)

        if not path.is_file():
            raise IsADirectoryError(path)

        path.unlink()

        return f"Deleted file: {path}"

    #^ ==========================================================
    #^ 2) Directory Related Tools
    #^ ==========================================================

    def create_directory(self, path: str, name: str) -> str:
        """Create a directory.

        Parent directories are created automatically.

        Args:
            path: Parent directory.
            name: Name of the new directory.

        Returns:
            Status message.
        """
        directory = Path(path) / name

        directory.mkdir(parents=True, exist_ok=True)

        return f"Directory created: {directory}"

    def delete_directory(self, path: str) -> str:
        """Delete a directory recursively.

        Args:
            path: Directory path.

        Returns:
            Status message.
        """
        directory = Path(path)

        if not directory.exists():
            raise FileNotFoundError(directory)

        if not directory.is_dir():
            raise NotADirectoryError(directory)

        shutil.rmtree(directory)

        return f"Directory deleted: {directory}"

    #^ ==========================================================
    #^ 3) Other File & Directory Operations
    #^ ==========================================================

    def move(self, source: str, destination: str) -> str:
        """Move a file or directory.

        Args:
            source: Existing file or directory.
            destination: Destination path.

        Returns:
            Status message.
        """
        src = Path(source)
        dst = Path(destination)

        shutil.move(str(src), str(dst))

        return f"Moved '{src}' -> '{dst}'"

    def copy(self, source: str, destination: str) -> str:
        """Copy a file or directory.

        Directories are copied recursively.

        Args:
            source: Existing file or directory.
            destination: Destination path.

        Returns:
            Status message.
        """
        src = Path(source)
        dst = Path(destination)

        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

        return f"Copied '{src}' -> '{dst}'"

    def rename(self, path: str, new_name: str) -> str:
        """Rename a file or directory.

        Args:
            path: Existing file or directory.
            new_name: New name only.

        Returns:
            Status message.
        """
        src = Path(path)

        if not src.exists():
            raise FileNotFoundError(src)

        dst = src.with_name(new_name)

        src.rename(dst)

        return f"Renamed '{src}' -> '{dst}'"

    def exists(self, path: str) -> bool:
        """Check whether a file or directory exists.

        Args:
            path: File or directory path.

        Returns:
            True if the path exists, otherwise False.
        """
        return Path(path).exists()

    def list_directories(self, path: str) -> List[str]:
        """List immediate subdirectories.

        Args:
            path: Parent directory.

        Returns:
            List of directory paths.
        """
        directory = Path(path)

        if not directory.exists():
            raise FileNotFoundError(directory)

        return [str(item) for item in directory.iterdir() if item.is_dir()]

    def search(self, path: str, pattern: str = "*") -> List[str]:
        """Recursively search using a glob pattern.

        Examples:
            *.py
            *.txt
            *

        Args:
            path: Root directory.
            pattern: Glob pattern.

        Returns:
            List of matching paths.
        """
        directory = Path(path)

        if not directory.exists():
            raise FileNotFoundError(directory)

        return [str(item) for item in directory.rglob(pattern)]

    def get_metadata(self, path: str) -> Dict[str, Any]:
        """Get filesystem metadata.

        Args:
            path: File or directory path.

        Returns:
            Dictionary containing metadata.
        """
        p = Path(path)

        if not p.exists():
            raise FileNotFoundError(p)

        stat = p.stat()

        return {
            "name": p.name,
            "path": str(p.resolve()),
            "type": "directory" if p.is_dir() else "file",
            "size": stat.st_size,
            "created": stat.st_ctime,
            "modified": stat.st_mtime,
            "accessed": stat.st_atime,
            "suffix": p.suffix,
            "parent": str(p.parent),
            "absolute": str(p.absolute()),
        }

    def open_file(self, path: str) -> str:
        """Open a file using the system's default application.

        Args:
            path: File path.

        Returns:
            Status message.
        """
        file = Path(path)

        if not file.exists():
            raise FileNotFoundError(file)

        if not file.is_file():
            raise IsADirectoryError(file)

        try:
            os.startfile(file)
            return f"Opened file: {file}"
        except AttributeError:
            return "Opening files is supported only on Windows."

    def open_directory(self, path: str) -> str:
        """Open a directory in the system file explorer.

        Args:
            path: Directory path.

        Returns:
            Status message.
        """
        directory = Path(path)

        if not directory.exists():
            raise FileNotFoundError(directory)

        if not directory.is_dir():
            raise NotADirectoryError(directory)

        try:
            os.startfile(directory)
            return f"Opened directory: {directory}"
        except AttributeError:
            return "Opening directories is supported only on Windows."

In [ ]:
file_op=FileManager()

In [ ]:
file_op.create_directory("./","new_dir")

'Directory created: new_dir'

In [ ]:
file_op.create_file("./new_dir/fist.py")

'File created successfully: new_dir\\fist.py'

In [ ]:
file_op.open_file("./new_dir/fist.py")

'Opened file: new_dir\\fist.py'

In [ ]:
file_op.open_directory('./new_dir')

'Opened directory: new_dir'

In [ ]:
file_op.get_metadata('./new_dir/fist.py')

{'name': 'fist.py',
 'path': 'D:\\Windows_Agent\\notebooks\\new_dir\\fist.py',
 'type': 'file',
 'size': 0,
 'created': 1782839711.2113569,
 'modified': 1782839711.2113569,
 'accessed': 1782839711.2113569,
 'suffix': '.py',
 'parent': 'new_dir',
 'absolute': 'd:\\Windows_Agent\\notebooks\\new_dir\\fist.py'}

In [ ]:
file_op.list_directories(r'D:\\')

['D:\\$RECYCLE.BIN',
 'D:\\.xinstall',
 'D:\\Agentic AI',
 'D:\\AI Agent',
 'D:\\AI Short Video Ads Generator',
 'D:\\AI+Agentic+Applications',
 'D:\\Android Studio',
 'D:\\BackEnd',
 'D:\\BERT',
 'D:\\Books',
 'D:\\ComputerVision',
 'D:\\ContextEngineering',
 'D:\\CSS',
 'D:\\cursor',
 'D:\\DataSmith',
 'D:\\DeepSeek',
 'D:\\Devin',
 'D:\\Docker',
 'D:\\FastAPI-BackEnd',
 'D:\\FlutterDev',
 'D:\\GenAI+Job+Preperation Web App',
 'D:\\JavaScripts',
 'D:\\LangChain',
 'D:\\LangGraph',
 'D:\\LocalSend',
 'D:\\MCP',
 'D:\\MCP_Deploy',
 'D:\\Medical+RAG+Chatbot',
 'D:\\new_dir',
 'D:\\Node JS',
 'D:\\Pitch_Visualizer',
 'D:\\Program Files',
 'D:\\Project Idea',
 'D:\\PythonSnakeGame',
 'D:\\React',
 'D:\\Reserach Paper',
 'D:\\SARVAKSH',
 'D:\\SarvakshApp',
 'D:\\Scientech 2137 V2',
 'D:\\sql',
 'D:\\SQL2022',
 'D:\\SteamLibrary',
 'D:\\System Volume Information',
 'D:\\TailwindCSS',
 'D:\\Trae',
 'D:\\TypeScripts',
 'D:\\Video Calling Interview Platform',
 'D:\\WebSocket',
 'D:\\Windows_Ag

In [ ]:
file_op.write_file('./new_dir/fist.py',"print('Hello Word')\n")

'Written successfully: new_dir\\fist.py'

In [ ]:
file_op.append_file('./new_dir/fist.py',"print(\"this is appending \")")

'Appended successfully: new_dir\\fist.py'

In [ ]:
file_op.exists(r'D:\Devin\resources\app\node_modules')

True

In [ ]:
file_op.read_file('./new_dir/fist.py')

'print(\'Hello Word\')\nprint("this is appending ")'

In [ ]:
file_op.search(r"D:\Devin\resources\app\node_modules",'d3')

['D:\\Devin\\resources\\app\\node_modules\\d3',
 'D:\\Devin\\resources\\app\\node_modules\\@types\\d3']

In [ ]:
file_op.rename('./new_dir/fist.py','new_file.py')

"Renamed 'new_dir\\fist.py' -> 'new_dir\\new_file.py'"

In [ ]:
file_op.delete_file('./new_dir/new_file.py')

'Deleted file: new_dir\\new_file.py'

In [ ]:
file_op.delete_directory('./new_dir')

'Directory deleted: new_dir'

In [7]:
import inspect

In [8]:
methods=inspect.getmembers(
    FileManager,
    predicate=inspect.isfunction
)

In [11]:
methods

[('__init__', <function __main__.FileManager.__init__(self) -> None>),
 ('append_file',
  <function __main__.FileManager.append_file(self, file_name: str, content) -> str>),
 ('copy',
  <function __main__.FileManager.copy(self, source: str, destination: str) -> str>),
 ('create_directory',
  <function __main__.FileManager.create_directory(self, path: str, name: str) -> str>),
 ('create_file',
  <function __main__.FileManager.create_file(self, file_name: str) -> str>),
 ('delete_directory',
  <function __main__.FileManager.delete_directory(self, path: str) -> str>),
 ('delete_file',
  <function __main__.FileManager.delete_file(self, file_name: str) -> str>),
 ('execute',
  <function __main__.FileManager.execute(self, action: str, **kwargs)>),
 ('exists', <function __main__.FileManager.exists(self, path: str) -> bool>),
 ('get_metadata',
  <function __main__.FileManager.get_metadata(self, path: str) -> Dict[str, Any]>),
 ('list_directories',
  <function __main__.FileManager.list_director

In [16]:
name,func=methods[2] 

print(name)
print(inspect.signature(func))
print(inspect.getdoc(func))
print(func.__annotations__)

copy
(self, source: str, destination: str) -> str
Copy a file or directory.

Directories are copied recursively.

Args:
    source: Existing file or directory.
    destination: Destination path.

Returns:
    Status message.
{'source': <class 'str'>, 'destination': <class 'str'>, 'return': <class 'str'>}


In [22]:
import inspect
from typing import Any, Dict, Type


def build_tool_schema(tool_class: Type) -> Dict[str, Any]:
    """
    Generate a tool schema from a class using its public methods,
    type hints, default values, and docstrings.

    Args:
        tool_class: Tool class (not an instance).

    Returns:
        Dictionary containing the tool schema.
    """

    def build_action_schema(func):
        signature = inspect.signature(func)

        parameters = {}

        for name, param in signature.parameters.items():

            # Ignore internal parameters
            if name in ("self", "action"):
                continue

            if param.kind == inspect.Parameter.VAR_KEYWORD:
                continue

            annotation = (
                param.annotation.__name__
                if param.annotation != inspect._empty
                and hasattr(param.annotation, "__name__")
                else (
                    str(param.annotation)
                    if param.annotation != inspect._empty
                    else "Any"
                )
            )

            parameters[name] = {
                "type": annotation,
                "required": param.default == inspect._empty,
                "default": (
                    None
                    if param.default == inspect._empty
                    else param.default
                ),
            }

        return_annotation = func.__annotations__.get("return", None)

        if hasattr(return_annotation, "__name__"):
            return_annotation = return_annotation.__name__
        elif return_annotation is None:
            return_annotation = "None"
        else:
            return_annotation = str(return_annotation)

        return {
            "name": func.__name__,
            "description": inspect.getdoc(func) or "",
            "parameters": parameters,
            "returns": return_annotation,
        }

    schema = {
        "tool": tool_class.__name__,
        "description": inspect.getdoc(tool_class) or "",
        "actions": [],
    }

    for name, func in inspect.getmembers(
        tool_class,
        predicate=inspect.isfunction,
    ):

        # Ignore private/internal methods
        if name.startswith("_"):
            continue

        # Ignore dispatcher
        if name == "execute":
            continue

        schema["actions"].append(
            build_action_schema(func)
        )

    return schema

In [23]:
build_tool_schema(FileManager)

{'tool': 'FileManager',
 'description': 'Tools for managing local files and directories.\n\nAll paths may be absolute or relative to the current working directory.',
 'actions': [{'name': 'append_file',
   'description': 'Append data to a file.\n\nCreates the file if it does not exist.\n\nArgs:\n    file_name: File path.\n    content: Text (str) or binary (bytes) data.\n\nReturns:\n    Status message.',
   'parameters': {'file_name': {'type': 'str',
     'required': True,
     'default': None},
    'content': {'type': 'Any', 'required': True, 'default': None}},
   'returns': 'str'},
  {'name': 'copy',
   'description': 'Copy a file or directory.\n\nDirectories are copied recursively.\n\nArgs:\n    source: Existing file or directory.\n    destination: Destination path.\n\nReturns:\n    Status message.',
   'parameters': {'source': {'type': 'str', 'required': True, 'default': None},
    'destination': {'type': 'str', 'required': True, 'default': None}},
   'returns': 'str'},
  {'name': '